# StudentInsight Google Colab and Jupyter

The default staff workspace now uses **Kaggle xAPI-Edu-Data v6** for Low / Middle / High academic performance classification across 12 recorded subjects. The notebook trains and evaluates the same classifier, displays charts, predicts an individual example and exports its results. The UCI Mathematics and Portuguese grade models remain a separately labelled reference workflow.

Kaggle source: https://www.kaggle.com/datasets/aljarah/xAPI-Edu-Data — 480 historical LMS records, CC BY-SA 4.0. Authors: Elaf Abu Amrieh, Thair Hamtini and Ibrahim Aljarah. See data/kaggle/provenance.json for attribution and source documentation. No exact marks or verified prediction cutoff dates are supplied. These models support a demonstration, not validated local deployment.

**Workflow:** setup and optional Drive connection; Kaggle training and individual/class predictions; separately labelled UCI reference experiments; combined ZIP export.

**Supported Kaggle subjects:** Arabic, Biology, Chemistry, English, French, Geology, History, Information Technology, Mathematics, Quran, Science and Spanish. These use one pooled classifier with subject as an input.

[GitHub project](https://github.com/GamerStudioszw/Student-Performance-Prediction) | [Open current notebook in Colab](https://colab.research.google.com/github/GamerStudioszw/Student-Performance-Prediction/blob/main/notebooks/StudentInsight.ipynb)

Run cells in order. Google Drive is optional; cloud authorization and sharing require your Google account. A saved notebook or project ZIP does not grant anyone access.


## 1. Locate the project and prepare the notebook environment


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPOSITORY = "https://github.com/GamerStudioszw/Student-Performance-Prediction.git"
BRANCH = "main"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/Student-Performance-Prediction")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPOSITORY, str(PROJECT_ROOT)], check=True)
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next((path for path in (current, *current.parents)
                         if (path / "model.py").is_file()
                         and (path / "requirements.txt").is_file()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError("Open this notebook inside the StudentInsight repository.")

if not (PROJECT_ROOT / "model_service.py").is_file():
    raise RuntimeError("The selected folder is not the StudentInsight project.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Install into the current Colab kernel before importing numerical libraries.
# Local dependencies are installed by the launcher, avoiding repeated pip changes.
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)

print("Environment:", "Google Colab" if IN_COLAB else "Local Jupyter")
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)
if (PROJECT_ROOT / ".git").exists():
    print("Git revision:", subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


If package installation requests a runtime restart, restart the kernel/runtime and run all cells again. In Colab, an existing checkout is reused without overwriting your edits; start a fresh runtime to clone the latest GitHub version.


In [ ]:
import json
import altair as alt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, FileLink

from model import (SCHEMA_VERSION, PUBLIC_SUBJECTS, FEATURE_SETS, load_data,
                   train_and_evaluate, prediction_report, alert_evaluation)
from model_service import SOURCE, file_hash, finalize_bundle, save_bundle
from cohort import export_csv

if SCHEMA_VERSION < 3:
    raise RuntimeError("This checkout predates the predictor repairs. Use the current main branch.")
print("scikit-learn:", sklearn.__version__)
print("UCI reference source (used in section 3):", SOURCE)
datasets = {subject: load_data(PROJECT_ROOT / "data" / f"student-{subject}.csv")
            for subject in PUBLIC_SUBJECTS}
display(pd.DataFrame([{"Subject": PUBLIC_SUBJECTS[key], "Records": len(frame),
                       "Grade scale": "0-20", "Prediction cutoff": "After period two"}
                      for key, frame in datasets.items()]))


## 1a. Optional: connect Google Drive

Drive is enabled by default in Colab. Run the next cell and approve Google's account-access prompt. This uses Colab's standard Drive mount; no password, token or API key is stored in the notebook. Mounting grants notebook code access to your Drive, not just one folder. This workflow reads only the dataset filename you specify and writes reports under **My Drive / StudentInsight / reports** and project snapshots under **My Drive / StudentInsight / projects**. See [Google's Drive access explanation](https://research.google.com/colaboratory/faq.html).

Two folders are created: `StudentInsight/datasets` for your CSV files and `StudentInsight/reports` for result ZIPs. Existing files are not replaced. Set `USE_GOOGLE_DRIVE = False` to run without Drive. If authorization fails, the cell stops; retry or turn the setting off.

**Local Jupyter:** Colab's mount is unavailable. If you already have Google Drive for desktop, set `LOCAL_DRIVE_FOLDER` to an existing synced folder. Otherwise use the ordinary local exports. This does not automatically link your Streamlit login or deployed dashboard to Drive.


In [ ]:
from notebook_storage import prepare_drive_workspace, read_drive_cohort, save_drive_archive

USE_GOOGLE_DRIVE = IN_COLAB  # Colab asks you to authorize Drive; set False to skip.
LOCAL_DRIVE_FOLDER = ""  # Optional existing Drive-for-desktop folder in local Jupyter.
SAVE_REPORTS_TO_DRIVE = True  # Applies only after a successful connection.
DRIVE_WORKSPACE = None

if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Use the Colab hosted runtime, or set LOCAL_DRIVE_FOLDER in local Jupyter.")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_WORKSPACE = prepare_drive_workspace(Path("/content/drive/MyDrive"))
elif LOCAL_DRIVE_FOLDER:
    DRIVE_WORKSPACE = prepare_drive_workspace(Path(LOCAL_DRIVE_FOLDER))

if DRIVE_WORKSPACE is None:
    print("Drive is disabled. Reports will be saved locally for download.")
else:
    print("Connected workspace:", DRIVE_WORKSPACE)
    print("CSV input folder:", DRIVE_WORKSPACE / "datasets")
    print("Report destination:", DRIVE_WORKSPACE / "reports")


## 1b. Save the complete project and notebook to Drive

When Drive is connected, the next cell saves a new project folder containing the committed source ZIP, a clean notebook copy, file hashes and startup instructions. The ZIP includes the application, public datasets, tests, documentation, reports and slides. It excludes virtual environments, account databases, credentials, private uploads, ignored model binaries and untracked research downloads. Commit tracked edits before exporting.

Open **Google Drive > StudentInsight > projects** and share the new snapshot folder with your supervisor as **Viewer**. Verify the recipient in the sharing list. Saving the files does not itself grant access. The notebook copy opens in Colab; the Streamlit staff interface remains a separate application. Each export creates a new folder and preserves older snapshots.


In [ ]:
from project_backup import save_project_snapshot

SAVE_PROJECT_TO_DRIVE = True
if DRIVE_WORKSPACE is not None and SAVE_PROJECT_TO_DRIVE:
    project_snapshot = save_project_snapshot(PROJECT_ROOT, DRIVE_WORKSPACE / "projects")
    print("Complete committed project saved:", project_snapshot)
    print("Notebook:", project_snapshot / "StudentInsight.ipynb")
    print("Share this snapshot folder in Google Drive to grant supervisor access.")
else:
    print("Project backup skipped: connect Drive in section 1a to save the complete project.")


## 2. Train the default Kaggle performance model

Compare a most-frequent baseline, logistic regression and random forest using grouped five-fold validation macro F1. A fixed 20% group holdout is reserved for testing. Identical selected-input groups cannot cross partitions, but the dataset has no student IDs to exclude all repeated-student overlap. Gender, nationality, birthplace, parent variables and names are excluded from predictors.


In [ ]:
from kaggle_model import (load_data as load_kaggle_data, train_and_evaluate as train_kaggle,
                          FEATURES as KAGGLE_FEATURES, BANDS as KAGGLE_BANDS, SOURCE as KAGGLE_SOURCE)
from kaggle_cohort import analyse as analyse_kaggle, read_class as read_kaggle_class

kaggle_data = load_kaggle_data()
kaggle_model, kaggle_metrics, kaggle_test = train_kaggle(kaggle_data)
display(pd.DataFrame(kaggle_metrics["comparison"]).drop(columns="fold_scores"))
display(pd.DataFrame([{key: kaggle_metrics[key] for key in
    ["selected_model", "raw_rows", "unique_model_rows", "train_rows", "test_rows", "accuracy", "macro_f1", "balanced_accuracy"]}]))
display(pd.DataFrame(kaggle_metrics["confusion_matrix"], index=["L", "M", "H"], columns=["L", "M", "H"]))
counts = kaggle_test.predicted_band.value_counts().rename_axis("Band").reset_index(name="Records")
display(alt.Chart(counts).mark_bar().encode(x="Band:N", y="Records:Q", tooltip=["Band", "Records"]))
print(kaggle_metrics["limitations"])


## 2a. Predict an individual or an optional Kaggle class file

The individual example uses a held-out historical row. Names are labels only. For your own class, supply the Kaggle template columns described in the app Data guide. Optional `Class` outcomes never enter the predictor. `KAGGLE_CLASS_CSV` is read only from the connected StudentInsight/datasets folder; leave blank to skip. UCI class files have a different schema and cannot be substituted.


In [ ]:
kaggle_example = kaggle_test[KAGGLE_FEATURES].head(1).copy()
kaggle_example["student_name"] = "Historical demonstration row"
kaggle_example["student_id"] = ""
kaggle_example["class_name"] = "Notebook example"
kaggle_single = analyse_kaggle(kaggle_model, kaggle_example, kaggle_metrics)
display(kaggle_single)

KAGGLE_CLASS_CSV = ""
kaggle_class_report = None
if KAGGLE_CLASS_CSV:
    if DRIVE_WORKSPACE is None:
        raise RuntimeError("Connect Drive before choosing a class CSV.")
    directory = (DRIVE_WORKSPACE / "datasets").resolve()
    path = (directory / KAGGLE_CLASS_CSV).resolve()
    if not path.is_relative_to(directory) or path.suffix.lower() != ".csv":
        raise ValueError("Choose a CSV inside StudentInsight/datasets.")
    if path.stat().st_size > 2 * 1024 * 1024:
        raise ValueError("Use a CSV no larger than 2 MiB.")
    kaggle_class_report = analyse_kaggle(kaggle_model, read_kaggle_class(path.read_bytes()), kaggle_metrics)
    display(kaggle_class_report)


## UCI reference workflow

The following cells retain the earlier grade-regression experiment for comparison. UCI grade errors and intervals are a different task from Kaggle band classification. Do not combine their metrics or relabel their subjects.


## 3. UCI reference: train Mathematics and Portuguese

This explicitly runs a new experiment **in memory**, using the same six candidates, training folds, fixed seed, simplicity rule and separate calibration/test partitions as the application. It does not overwrite historical artifacts or change the deployed predictor.

The existing test results already informed the repairs: these comparisons are **not independent confirmation**. The G2 persistence baseline is allowed to win. All required historical values must be recorded; missing inputs are never invented or imputed.


In [ ]:
bundles = {}
for subject, frame in datasets.items():
    path = PROJECT_ROOT / "data" / f"student-{subject}.csv"
    bundles[subject] = finalize_bundle(
        *train_and_evaluate(frame), subject=subject,
        fingerprint=file_hash(path), source=SOURCE)

summary = pd.DataFrame([
    {"Subject": PUBLIC_SUBJECTS[subject], "Predictor": metrics["selected_model"],
     "Feature set": metrics["feature_set"], "Required inputs": ", ".join(metrics["features"]),
     "Test MAE": metrics["mae"], "Test RMSE": metrics["rmse"], "Test R2": metrics["r2"],
     "G2 baseline MAE": metrics["g2_baseline_mae"],
     "Nominal coverage": metrics["interval"]["nominal_coverage"],
     "Observed test coverage": metrics["interval"]["test_coverage"],
     "Average interval width": metrics["interval"]["average_width"],
     "Calibration records": metrics["calibration_rows"], "Test records": metrics["test_rows"]}
    for subject, (_, metrics, _) in bundles.items()
])
display(summary)


## 3a. UCI reference: candidate comparisons and charts

Set `SUBJECT` to `"mat"` or `"por"`, then rerun this cell and the cells below it. The comparison table uses cross-validation MAE on fitting records. The chart uses the separate historical test partition. MAE/RMSE are grade-point errors; R2 is not an accuracy percentage.


In [ ]:
SUBJECT = "mat"  # "mat": Mathematics; "por": Portuguese language
if SUBJECT not in bundles:
    raise ValueError("Choose mat or por; other subjects require suitable completed history.")
fitted, metrics, test_results = bundles[SUBJECT]
print(PUBLIC_SUBJECTS[SUBJECT], "|", metrics["predictor_kind"])
print("Selection rule:", metrics["selection_rule"])
comparison = pd.DataFrame(metrics["comparison"])
display(comparison[["model", "feature_set", "features", "cv_mae", "cv_std"]])
display(alt.Chart(comparison).mark_bar().encode(
    x=alt.X("cv_mae:Q", title="Cross-validation MAE (grade points)"),
    y=alt.Y("model:N", sort="x", title="Candidate"),
    tooltip=["model", "feature_set", "cv_mae", "cv_std"]
).properties(title=f"{PUBLIC_SUBJECTS[SUBJECT]}: validation comparison", height=230))

display(alt.Chart(test_results).mark_circle(size=65, opacity=0.5).encode(
    x=alt.X("actual_grade:Q", title="Recorded final grade G3"),
    y=alt.Y("predicted_grade:Q", title="Predicted final grade G3"),
    tooltip=["source_row", "actual_grade", "predicted_grade", "interval_lower", "interval_upper"]
).properties(title="Historical test records: recorded versus predicted", height=300))

# Changing subject invalidates any previously loaded class report.
drive_class_report = None
drive_class_bundle_id = None


## 3b. UCI reference: intervals and below-reference alerts

Intervals use the finite-sample split-conformal absolute-error quantile. A single calibration radius gives **constant width**, not personalized uncertainty. Raw predictions and intervals are not clipped. An 80% nominal coverage target is not an individual's probability of passing. Realized coverage can differ from the target.

For alerts, the positive outcome is **recorded G3 < reference grade** and the alert is **predicted G3 < the same reference**. Equality is not below reference. Undefined scores are displayed as missing, not zero. These are historical test metrics, separate from uploaded class performance.


In [ ]:
REFERENCE_GRADE = 10
interval = metrics["interval"]
display(pd.DataFrame([interval]))
alerts = alert_evaluation(test_results.actual_grade, test_results.predicted_grade, REFERENCE_GRADE)
display(pd.DataFrame([alerts]))
display(pd.DataFrame(
    [[alerts["tp"], alerts["fn"]], [alerts["fp"], alerts["tn"]]],
    index=["Recorded G3 below reference", "Recorded G3 at/above reference"],
    columns=["Alert", "No alert"]))


## 3c. UCI reference: predict from recorded inputs

Use a real bundled held-out row as an example. Only the selected predictor's required columns are passed; optional G1 is retained for observed progress. G3 is never passed to prediction. To try your own profile, replace `profile` with a one-row DataFrame containing the displayed required inputs, using the same whole-number 0-20 grade scale.

G2 - G1 describes recorded progress, not independent information or a causal effect. Percentage equivalent means grade multiplied by five, not probability of passing.


In [ ]:
print("Required prediction inputs:", metrics["features"])
source_row = int(test_results.iloc[0]["source_row"])
profile_columns = list(dict.fromkeys([*metrics["features"], "G1"]))
profile = datasets[SUBJECT].loc[[source_row], profile_columns].copy()
report = prediction_report(fitted, profile, metrics, threshold=REFERENCE_GRADE)
display(report)


## 3d. Optional UCI reference: predict a class from Drive

Place a UTF-8 CSV in **My Drive / StudentInsight / datasets**, then enter its filename below. Its records must be for the selected subject and use that subject's 0-20 grade scale. Include `student_name` or `student_id` (or both), `class_name` and every selected predictor input printed below. Students sharing a name in the same class need distinct IDs. Names label reports and never enter the predictor. G1 is optional for progress when the predictor only requires G2; G3 is optional for comparison with recorded outcomes. Missing required inputs are rejected, never fabricated.

A blank filename skips this step. This reads the file without modifying it and predicts using the already selected subject model; it does not retrain on the class or mix its results with the historical test evaluation. The class predictions are included separately in the experiment ZIP. Rerun this cell after changing the subject/model. Only load records you intend to process in this notebook; clear saved cell outputs before sharing it.


In [ ]:
from cohort import analyse_cohort

DRIVE_CLASS_CSV = ""  # Example filename: mathematics_class.csv
# Clear earlier results before attempting another file.
drive_class_report = None
drive_class_bundle_id = None
print("Required CSV columns:", ["student_name or student_id (or both)", "class_name", *metrics["features"]])
if DRIVE_CLASS_CSV:
    if DRIVE_WORKSPACE is None:
        raise RuntimeError("Connect Drive in section 1a before reading a Drive CSV.")
    drive_class = read_drive_cohort(DRIVE_WORKSPACE, DRIVE_CLASS_CSV, metrics["features"])
    drive_class_report = analyse_cohort(fitted, drive_class, metrics, REFERENCE_GRADE)
    drive_class_bundle_id = metrics["bundle_id"]
    print("Drive class predictions:", len(drive_class_report), "records for", PUBLIC_SUBJECTS[SUBJECT])
    display(drive_class_report)
else:
    print("No Drive class selected; continuing with the bundled historical demonstration.")


## 4. Export Kaggle and UCI experiment results

Create a new folder under `notebook-results/` (excluded from Git). The `kaggle/` folder contains evaluation metrics, held-out predictions, an individual prediction and any loaded Kaggle class report. Separate UCI files contain both reference subjects' evaluations, the UCI summary, selected-subject profile and alerts. Environment provenance records the Git revision and tracked-change status at export time, alongside package versions. A modified checkout must not be described as identical to its recorded commit. It does not replace saved application bundles. `FileLink` works in local Jupyter; set `DOWNLOAD_IN_COLAB = True` to download a ZIP in Colab. If a Drive class report was loaded for this active bundle, it is included as a separate class-prediction CSV. Local exports remain available even if the later Drive copy fails.


In [ ]:
from datetime import datetime, timezone
import importlib.metadata
import shutil
import uuid

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
export_dir = PROJECT_ROOT / "notebook-results" / run_id
export_dir.mkdir(parents=True, exist_ok=False)
kaggle_dir = export_dir / "kaggle"
kaggle_dir.mkdir()
(kaggle_dir / "metrics.json").write_text(json.dumps(kaggle_metrics, indent=2) + "\n", encoding="utf-8")
(kaggle_dir / "test_predictions.csv").write_bytes(export_csv(kaggle_test))
(kaggle_dir / "single_prediction.csv").write_bytes(export_csv(kaggle_single))
if kaggle_class_report is not None:
    (kaggle_dir / "class_report.csv").write_bytes(export_csv(kaggle_class_report))
summary.to_csv(export_dir / "summary.csv", index=False)
for subject, (_, subject_metrics, results) in bundles.items():
    (export_dir / f"{subject}_metrics.json").write_text(
        json.dumps(subject_metrics, indent=2, allow_nan=False), encoding="utf-8")
    results.to_csv(export_dir / f"{subject}_test_predictions.csv", index=False)
(export_dir / f"{SUBJECT}_profile.csv").write_bytes(export_csv(report))
pd.DataFrame([{"subject": SUBJECT, "bundle_id": metrics["bundle_id"],
               "evaluation_population": "Historical test partition", **alerts}]).to_csv(
    export_dir / f"{SUBJECT}_alerts.csv", index=False)
git_revision = None
git_tracked_changes = None
if (PROJECT_ROOT / ".git").exists():
    git_revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
    git_tracked_changes = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        cwd=PROJECT_ROOT, text=True).strip())
provenance = {"git_revision": git_revision, "git_tracked_changes": git_tracked_changes,
              "python": sys.version, "environment": "Colab" if IN_COLAB else "Local Jupyter",
              "packages": {name: importlib.metadata.version(name)
                           for name in ["numpy", "pandas", "scikit-learn", "joblib", "altair"]}}
(export_dir / "environment.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
if drive_class_report is not None:
    if drive_class_bundle_id != metrics["bundle_id"]:
        raise ValueError("Reload the Drive class using the selected model before exporting.")
    (export_dir / f"{SUBJECT}_drive_class_predictions.csv").write_bytes(export_csv(drive_class_report))
archive = Path(shutil.make_archive(str(export_dir), "zip", root_dir=export_dir))
print("Saved experiment:", export_dir)
DOWNLOAD_IN_COLAB = False
if IN_COLAB:
    print("ZIP:", archive, "- set DOWNLOAD_IN_COLAB = True to download.")
    if DOWNLOAD_IN_COLAB:
        from google.colab import files
        files.download(str(archive))
else:
    display(FileLink(archive.relative_to(PROJECT_ROOT).as_posix(), url_prefix="/files/"))


## 4a. Save the ZIP to Drive

When Drive is connected and `SAVE_REPORTS_TO_DRIVE` is true, this copies the completed ZIP into `StudentInsight/reports` and checks that the copied bytes match. Each experiment has a unique name; an existing report is never overwritten. If copying fails or the file already exists, the local ZIP remains available. Run the export cell again to create a new experiment filename. Drive reports may include the private class predictions you chose to load.


In [ ]:
if DRIVE_WORKSPACE is not None and SAVE_REPORTS_TO_DRIVE:
    drive_archive = save_drive_archive(archive, DRIVE_WORKSPACE)
    print("Report ZIP saved and verified on Drive:", drive_archive)
else:
    print("Drive copy skipped. Your experiment ZIP is available locally:", archive)


## 5. Optional UCI reference: save a local application bundle

This cell saves UCI grade models only. The Kaggle staff app trains and caches its classifier from the bundled CSV; `python train.py` regenerates Kaggle evaluation files. Leave `SAVE_APPLICATION_BUNDLE = False` for exploration. Setting it to `True` saves the selected subject as a new immutable run and updates its local `latest.json` pointer using the same artifact service as `train.py`. Repeat the training cell before saving the same bundle a second time. In Colab, this affects only the temporary runtime checkout; it does not deploy a model to your PC or the hosted app. Model binaries are excluded from Git.

For additional subjects, use `subject_training.train_subject` with genuine completed history and the chosen versioned input schema. Do not relabel Mathematics/Portuguese models or fill missing historical parameters. The Halabja download is not integrated.


In [ ]:
SAVE_APPLICATION_BUNDLE = False
if SAVE_APPLICATION_BUNDLE:
    saved_directory = save_bundle(bundles[SUBJECT])
    print("Saved new application bundle:", saved_directory)
else:
    print("Application artifact pointers unchanged.")


## 6. Get updates and share the notebook

The Colab link loads the notebook from GitHub `main`. A copy saved in Drive is a separate file and does not automatically receive later GitHub edits. Reopen the GitHub Colab link for the latest notebook; save your own changes before replacing a copy. Use a fresh Colab runtime to load the latest project code.

For local Jupyter, save your work, close the notebook and run `git pull --ff-only` from the repository terminal. If Git reports local changes or divergent history, resolve them before continuing; do not discard work to force an update. Reopen the notebook and restart the kernel before running it again.

To grant supervisor access, share the intended Drive notebook or project snapshot with **czano@buse.ac.zw** as **Viewer** and verify the permission list. Committing or pushing to GitHub does not grant access to private Drive files. Clear private outputs before sharing a notebook.
